[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/089.%20The%20Single%20vs.%20Dual%20Sourcing%20Problem/P89-Tier-7_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 89. The Single vs. Dual Sourcing Problem

## Tier 7: Human–AI Symbiotic Partnership

### Purpose and scope

This notebook models sourcing recommendations as part of a human–AI decision process. An explainable scoring engine evaluates single-sourcing and, for sufficiently critical components, dual-sourcing options. A simulated human expert may accept, modify, override, or escalate a recommendation. Outcomes then update the model's performance records and selected expert attributes.

The example is a 12-week MediTech simulation with three components, four suppliers, and three experts. Inputs include component criticality, inventory, safety stock, demand forecasts, supplier cost, reliability, quality, capacity utilization, and weekly market conditions. Outputs include recommendations, collaborative decisions, simulated outcomes, partnership metrics, visualizations, and a report.

### Model assumptions

- Supplier and component attributes are treated as known inputs; this is a demonstration model, not a calibrated procurement system.
- Supplier scores and recommendation weights are illustrative. The model does not estimate them from historical data.
- The example simulates outcomes and human responses; it does not establish causal effects or demonstrate real-world cost savings, service improvements, or risk reduction.
- Dual sourcing uses a fixed 70%/30% allocation. It is considered only when component criticality is greater than 0.7 and at least two suppliers are available.
- Supplier capacity is represented by utilization in a score and risk calculation, not by a hard capacity constraint. Lead times, order quantities, contractual terms, inventory dynamics, and correlated supplier disruptions are outside the model.
- The AI score is a weighted decision heuristic, not a mathematical optimization model. Results depend on the stated scoring rules and simulated inputs.
- Random draws are used for market conditions, expert responses, and outcomes. No random seed is imposed, so each run can differ.

### Decision process

1. Analyze a component and the current market conditions.
2. Score suppliers and construct eligible sourcing options.
3. Evaluate options using weighted cost, service, and risk scores; retain the highest-scoring option.
4. Calculate a confidence score and provide a textual explanation of the recommendation.
5. Apply the intervention protocol: an expert reviews the recommendation when confidence or risk crosses a threshold, or when the trust rule calls for review.
6. Record a simulated outcome and update performance measures and expert attributes.

### Scoring model

For supplier $j$, the supplier score is

$$
S_j = 0.30 C_j + 0.25 Q_j + 0.30 R_j + 0.15 F_j,
$$

where $C_j = 1 - c_j / c_{\max}$ is the normalized cost score, $Q_j$ is quality, $R_j$ is reliability, and $F_j = 1 - u_j$ is capacity flexibility. Here, $c_j$ is unit cost, $c_{\max}$ is the largest supplier unit cost, and $u_j$ is capacity utilization. A higher score ranks a supplier more favorably; it is not itself a probability of success.

For option $o$, the decision engine evaluates

$$
V_o = w_c(1 - I_o) + w_s L_o + w_r(1 - R_o),
$$

where $I_o$ is the option's cost impact relative to the mean supplier cost, $L_o$ is its expected service level, and $R_o$ is its risk score. The initial weights are $w_c=0.4$, $w_s=0.4$, and $w_r=0.2$. The selected option is the one with the greatest $V_o$; ties retain the first option encountered. After enough recorded outcomes, the learning rule can adjust the cost and service weights when their recent prediction errors exceed a threshold, then renormalizes all three weights.

The candidate set contains the highest-ranked supplier as a single-source option. If the component is sufficiently critical, the two highest-ranked suppliers also form a dual-source option using a 70%/30% allocation. A switch option is added when the current supplier's risk is high and an alternative is available. These rules define the candidate set; the weighted score selects among its members.

### Confidence and human intervention

Confidence combines an assumed data-quality factor, a market-condition adjustment, a risk adjustment, and a historical-performance adjustment. It is clipped to the interval from 0.1 to 0.99. An expert review is triggered by confidence below 0.6, risk above 0.7, or the trust-based intervention rule implemented in the code. Expert behavior and simulated outcomes are stochastic; therefore, a run's results should be interpreted as an illustration of the process, not as a forecast.

### What to examine

- How the recommendation's explanation relates to its score, service estimate, cost impact, and risk assessment.
- How expert intervention and the final selected suppliers affect the reported decision.
- How trust, collaboration, decision quality, intervention frequency, learning progress, and the illustrative ROI score are calculated.
- Which conclusions are supported by the simulated records and which would require real, validated data.

In [ ]:
from collections import defaultdict
from dataclasses import dataclass
from datetime import datetime
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class HumanExpert:
    """Expert attributes used by the simulated intervention protocol."""
    expert_id: str
    name: str
    expertise_domain: str
    experience_years: int
    confidence_level: float
    decision_style: str
    trust_in_ai: float
    intervention_threshold: float
    collaboration_score: float


@dataclass
class AIRecommendation:
    """Recommendation, explanation, and projected outcomes from the AI engine."""
    recommendation_id: str
    component_name: str
    action_type: str
    primary_supplier: Optional[str]
    secondary_supplier: Optional[str]
    allocation_ratio: Optional[float]
    confidence_score: float
    reasoning: List[str]
    supporting_data: Dict[str, float]
    risk_assessment: float
    expected_outcome: Dict[str, float]
    created_at: datetime


@dataclass
class HumanIntervention:
    """Record of an expert's response to an AI recommendation."""
    intervention_id: str
    expert_id: str
    recommendation_id: str
    intervention_type: str
    reasoning: str
    modified_action: Optional[str]
    modified_supplier: Optional[str]
    confidence_in_decision: float
    timestamp: datetime
    outcome_impact: Optional[float] = None


@dataclass
class PartnershipMetrics:
    """Aggregate indicators describing the simulated partnership."""
    timestamp: datetime
    trust_level: float
    collaboration_effectiveness: float
    decision_quality_score: float
    intervention_frequency: float
    intervention_success_rate: float
    learning_progress: float
    roi_score: float


@dataclass
class SourcingDecision:
    """Final sourcing action after the recommendation and review process."""
    decision_id: str
    component_name: str
    final_action: str
    selected_suppliers: List[str]
    allocation_ratios: List[float]
    ai_recommendation: AIRecommendation
    human_intervention: Optional[HumanIntervention]
    decision_confidence: float
    expected_cost: float
    expected_service_level: float
    risk_mitigation: float
    made_at: datetime

In [ ]:
class ExplainableAIEngine:
    """Rank sourcing options and retain the reasoning behind each recommendation."""

    def __init__(self, components: List[Dict], suppliers: List[Dict]):
        if not suppliers:
            raise ValueError("At least one supplier is required.")
        self.components = components
        self.suppliers = suppliers
        self.decision_history = []
        self.performance_metrics = defaultdict(list)
        self.cost_weight = 0.4
        self.service_weight = 0.4
        self.risk_weight = 0.2
        self.learning_rate = 0.1
        self.adaptation_factor = 0.05

    def generate_recommendation(self, component_name: str, market_conditions: Dict) -> AIRecommendation:
        """Analyze a component, rank eligible options, and explain the selected option."""
        component = next((item for item in self.components if item["name"] == component_name), None)
        if component is None:
            raise ValueError("Component {!r} was not found.".format(component_name))

        analysis = self._analyze_sourcing_situation(component, market_conditions)
        options = self._generate_decision_options(component, analysis)
        best_option = self._evaluate_options(options)
        reasoning = self._generate_reasoning(best_option, analysis)
        confidence = self._calculate_confidence(best_option, analysis)

        recommendation = AIRecommendation(
            recommendation_id="ai_rec_{}".format(len(self.decision_history)),
            component_name=component_name,
            action_type=best_option["action_type"],
            primary_supplier=best_option["primary_supplier"],
            secondary_supplier=best_option["secondary_supplier"],
            allocation_ratio=best_option["allocation_ratio"],
            confidence_score=confidence,
            reasoning=reasoning,
            supporting_data=analysis["supporting_data"],
            risk_assessment=best_option["risk_score"],
            expected_outcome=best_option["expected_outcome"],
            created_at=datetime.now(),
        )
        self.decision_history.append(recommendation)
        return recommendation

    def _analyze_sourcing_situation(self, component: Dict, market_conditions: Dict) -> Dict:
        """Collect the component, market, and supplier information used by the scoring rules."""
        analysis = {
            "component_criticality": component["criticality_score"],
            "current_inventory": component["current_inventory"],
            "demand_forecast": component["demand_forecast"],
            "market_condition": market_conditions.get("market_index", 0.0),
            "disruption_risk": market_conditions.get("disruption_risk", 0.1),
            "supporting_data": {},
        }
        supplier_analysis = {}
        for supplier in self.suppliers:
            supplier_analysis[supplier["name"]] = {
                "cost": supplier["unit_cost"],
                "reliability": supplier["reliability_score"],
                "quality": supplier["quality_score"],
                "capacity_utilization": supplier["capacity_utilization"],
                "risk_score": self._calculate_supplier_risk(supplier, market_conditions),
                "overall_score": self._calculate_supplier_score(supplier),
            }
        analysis["supplier_analysis"] = supplier_analysis
        analysis["supporting_data"] = {
            "cost_efficiency_need": 1.0 - component["current_inventory"] / max(1.0, component["safety_stock"]),
            "service_level_requirement": component["criticality_score"],
            "risk_tolerance": 1.0 - component["criticality_score"],
            "market_volatility": abs(market_conditions.get("market_index", 0.0)),
        }
        return analysis

    def _generate_decision_options(self, component: Dict, analysis: Dict) -> List[Dict]:
        """Construct the single-source, eligible dual-source, and eligible switch options."""
        supplier_analysis = analysis["supplier_analysis"]
        ranked = sorted(
            supplier_analysis.items(), key=lambda item: item[1]["overall_score"], reverse=True
        )
        best_supplier = ranked[0][0]
        options = [{
            "action_type": "single_sourcing",
            "primary_supplier": best_supplier,
            "secondary_supplier": None,
            "allocation_ratio": None,
            "risk_score": supplier_analysis[best_supplier]["risk_score"],
            "expected_outcome": self._calculate_outcome("single", best_supplier, None, analysis),
        }]

        if component["criticality_score"] > 0.7 and len(ranked) >= 2:
            primary, secondary = ranked[0][0], ranked[1][0]
            options.append({
                "action_type": "dual_sourcing",
                "primary_supplier": primary,
                "secondary_supplier": secondary,
                "allocation_ratio": 0.7,
                "risk_score": max(
                    supplier_analysis[primary]["risk_score"],
                    supplier_analysis[secondary]["risk_score"],
                ) * 0.6,
                "expected_outcome": self._calculate_outcome("dual", primary, secondary, analysis),
            })

        current_supplier = component.get("current_supplier", best_supplier)
        if (
            current_supplier in supplier_analysis
            and supplier_analysis[current_supplier]["risk_score"] > 0.6
            and len(ranked) > 1
        ):
            alternative = next((name for name, _ in ranked if name != current_supplier), None)
            if alternative is not None:
                options.append({
                    "action_type": "switch_supplier",
                    "primary_supplier": alternative,
                    "secondary_supplier": None,
                    "allocation_ratio": None,
                    "risk_score": supplier_analysis[alternative]["risk_score"],
                    "expected_outcome": self._calculate_outcome("single", alternative, None, analysis),
                })
        return options

    def _evaluate_options(self, options: List[Dict]) -> Dict:
        """Select the option with the greatest weighted cost, service, and risk score."""
        if not options:
            raise ValueError("At least one sourcing option is required.")
        best_option = None
        best_score = -float("inf")
        for option in options:
            outcome = option["expected_outcome"]
            cost_score = (1.0 - outcome["cost_impact"]) * self.cost_weight
            service_score = outcome["service_level"] * self.service_weight
            risk_score = (1.0 - option["risk_score"]) * self.risk_weight
            total_score = cost_score + service_score + risk_score
            if total_score > best_score:
                best_score = total_score
                best_option = option
                best_option["total_score"] = total_score
        return best_option

    def _generate_reasoning(self, option: Dict, analysis: Dict) -> List[str]:
        """Translate the selected option and its metrics into an auditable explanation."""
        reasoning = []
        if option["action_type"] == "single_sourcing":
            reasoning.append("Single sourcing from {} was selected.".format(option["primary_supplier"]))
            reasoning.append("Weighted option score: {:.3f}.".format(option["total_score"]))
        elif option["action_type"] == "dual_sourcing":
            share = option["allocation_ratio"]
            reasoning.append("Dual sourcing was selected for a component above the criticality threshold.")
            reasoning.append(
                "Allocation: {} ({:.0%}) and {} ({:.0%}).".format(
                    option["primary_supplier"], share,
                    option["secondary_supplier"], 1.0 - share,
                )
            )
            reasoning.append("The model applies a diversification adjustment to the option's risk score.")
        else:
            reasoning.append("Switching to {} was selected because current-supplier risk is high.".format(
                option["primary_supplier"]
            ))
        if analysis["component_criticality"] > 0.8:
            reasoning.append("High component criticality increases the importance of risk mitigation.")
        if analysis["market_condition"] < -0.3:
            reasoning.append("The market index is low, which the model treats as an unfavorable condition.")
        elif analysis["market_condition"] > 0.3:
            reasoning.append("The market index is high, which the model treats as a favorable condition.")
        if analysis["disruption_risk"] > 0.2:
            reasoning.append("The supplied disruption-risk indicator is above 0.2.")
        outcome = option["expected_outcome"]
        reasoning.append("Expected cost impact relative to mean supplier cost: {:+.1%}.".format(
            outcome["cost_impact"]
        ))
        reasoning.append("Expected service level: {:.1%}.".format(outcome["service_level"]))
        reasoning.append("Risk assessment: {:.2f} (0 = low, 1 = high).".format(option["risk_score"]))
        return reasoning

    def _calculate_confidence(self, option: Dict, analysis: Dict) -> float:
        """Combine assumed data quality, market conditions, risk, and observed performance."""
        data_confidence = 0.8
        market_adjustment = 1.0 - abs(analysis["market_condition"]) * 0.2
        risk_adjustment = 1.0 - option["risk_score"] * 0.3
        if len(self.decision_history) > 10:
            recent_performance = np.mean([
                decision.expected_outcome.get("actual_performance", 0.8)
                for decision in self.decision_history[-10:]
            ])
        else:
            recent_performance = 0.8
        confidence = data_confidence * market_adjustment * risk_adjustment * recent_performance
        return float(np.clip(confidence, 0.1, 0.99))

    def _calculate_supplier_risk(self, supplier: Dict, market_conditions: Dict) -> float:
        """Combine reliability, capacity utilization, market, and quality risk components."""
        reliability_risk = 1.0 - supplier["reliability_score"]
        capacity_risk = max(0.0, (supplier["capacity_utilization"] - 0.8) * 2.0)
        market_risk = (1.0 - market_conditions.get("market_index", 0.0)) * 0.2
        quality_risk = (1.0 - supplier["quality_score"]) * 0.3
        return float(np.clip(
            reliability_risk + capacity_risk + market_risk + quality_risk, 0.0, 1.0
        ))

    def _calculate_supplier_score(self, supplier: Dict) -> float:
        """Calculate the weighted supplier ranking score."""
        max_cost = max(item["unit_cost"] for item in self.suppliers)
        cost_score = 1.0 - supplier["unit_cost"] / max_cost
        capacity_score = 1.0 - supplier["capacity_utilization"]
        return (
            cost_score * 0.30
            + supplier["quality_score"] * 0.25
            + supplier["reliability_score"] * 0.30
            + capacity_score * 0.15
        )

    def _calculate_outcome(
        self, action_type: str, primary: str, secondary: Optional[str], analysis: Dict
    ) -> Dict:
        """Estimate relative cost impact, service, and risk mitigation for an option."""
        supplier_analysis = analysis["supplier_analysis"]
        mean_cost = np.mean([item["cost"] for item in supplier_analysis.values()])
        if action_type == "single":
            data = supplier_analysis[primary]
            cost_impact = (data["cost"] - mean_cost) / mean_cost
            service_level = data["reliability"] * data["quality"]
        else:
            primary_data = supplier_analysis[primary]
            secondary_data = supplier_analysis[secondary]
            average_cost = primary_data["cost"] * 0.7 + secondary_data["cost"] * 0.3
            cost_impact = (average_cost - mean_cost) / mean_cost
            weighted_reliability = (
                primary_data["reliability"] * 0.7 + secondary_data["reliability"] * 0.3
            )
            service_level = min(
                0.98,
                weighted_reliability * max(primary_data["quality"], secondary_data["quality"]) * 1.1,
            )
        return {
            "cost_impact": float(cost_impact),
            "service_level": float(service_level),
            "risk_mitigation": 1.0 - analysis["disruption_risk"] if action_type == "dual" else 0.8,
        }

    def learn_from_outcome(self, recommendation: AIRecommendation, actual_outcome: Dict) -> None:
        """Store prediction errors and adapt cost/service weights using the stated rule."""
        recommendation.expected_outcome["actual_cost"] = actual_outcome.get("actual_cost", 0.0)
        recommendation.expected_outcome["actual_service"] = actual_outcome.get("actual_service", 0.0)
        recommendation.expected_outcome["actual_performance"] = actual_outcome.get(
            "performance_score", 0.8
        )
        cost_error = abs(
            recommendation.expected_outcome["cost_impact"] - actual_outcome.get("cost_impact", 0.0)
        )
        service_error = abs(
            recommendation.expected_outcome["service_level"] - actual_outcome.get("service_level", 0.0)
        )
        self.performance_metrics["cost_error"].append(cost_error)
        self.performance_metrics["service_error"].append(service_error)
        self.performance_metrics["confidence_accuracy"].append(
            1.0 - abs(recommendation.confidence_score - actual_outcome.get("actual_confidence", 0.8))
        )
        if len(self.performance_metrics["cost_error"]) > 5:
            recent_cost_error = np.mean(self.performance_metrics["cost_error"][-5:])
            recent_service_error = np.mean(self.performance_metrics["service_error"][-5:])
            if recent_cost_error > 0.1:
                self.cost_weight *= 1.0 + self.learning_rate * 0.1
            if recent_service_error > 0.1:
                self.service_weight *= 1.0 + self.learning_rate * 0.1
            total_weight = self.cost_weight + self.service_weight + self.risk_weight
            self.cost_weight /= total_weight
            self.service_weight /= total_weight
            self.risk_weight /= total_weight

In [ ]:
class HumanAIPartnership:
    """Coordinate AI recommendations, expert review, and outcome tracking."""

    def __init__(self, components: List[Dict], suppliers: List[Dict], experts: List[Dict]):
        if not experts:
            raise ValueError("At least one human expert is required.")
        self.components = components
        self.suppliers = suppliers
        self.experts = self._initialize_experts(experts)
        self.ai_engine = ExplainableAIEngine(components, suppliers)
        self.partnership_history = []
        self.trust_metrics = defaultdict(list)
        self.collaboration_metrics = defaultdict(list)
        self.learning_loop = LearningLoop()
        self.decision_outcomes = []
        self.partnership_roi = []

    def _initialize_experts(self, experts_data: List[Dict]) -> Dict[str, HumanExpert]:
        """Build expert records from the example data."""
        experts = {}
        for item in experts_data:
            expert = HumanExpert(
                expert_id=item["id"], name=item["name"],
                expertise_domain=item["domain"], experience_years=item["experience"],
                confidence_level=item["confidence"], decision_style=item["style"],
                trust_in_ai=item.get("initial_trust", 0.5),
                intervention_threshold=item.get("intervention_threshold", 0.7),
                collaboration_score=0.5,
            )
            experts[expert.expert_id] = expert
        return experts

    def make_collaborative_decision(self, component_name: str, market_conditions: Dict) -> SourcingDecision:
        """Generate a recommendation, apply any expert response, and record the final decision."""
        recommendation = self.ai_engine.generate_recommendation(component_name, market_conditions)
        intervention_needed, expert = self._assess_intervention_need(recommendation)
        intervention = self._get_human_intervention(recommendation, expert) if intervention_needed else None
        action = recommendation.action_type
        suppliers = []
        allocations = []

        if intervention is not None:
            if intervention.intervention_type in ("modify", "override"):
                action = intervention.modified_action or recommendation.action_type
                if action == "dual_sourcing":
                    primary = intervention.modified_supplier or recommendation.primary_supplier
                    secondary = recommendation.secondary_supplier
                    if secondary is None or secondary == primary:
                        alternatives = [item["name"] for item in self.suppliers if item["name"] != primary]
                        secondary = alternatives[0] if alternatives else None
                    if secondary is not None:
                        suppliers = [primary, secondary]
                        share = recommendation.allocation_ratio or 0.7
                        allocations = [share, 1.0 - share]
                    else:
                        action = "single_sourcing"
                        suppliers, allocations = [primary], [1.0]
                else:
                    supplier = intervention.modified_supplier or recommendation.primary_supplier
                    suppliers, allocations = [supplier], [1.0]
            self._update_trust(expert, recommendation, intervention)

        if not suppliers:
            if recommendation.action_type == "dual_sourcing":
                share = recommendation.allocation_ratio
                suppliers = [recommendation.primary_supplier, recommendation.secondary_supplier]
                allocations = [share, 1.0 - share]
            else:
                suppliers = [recommendation.primary_supplier]
                allocations = [1.0]

        if any(supplier is None for supplier in suppliers):
            raise ValueError("Every selected supplier must be defined.")
        if not np.isclose(sum(allocations), 1.0):
            raise ValueError("Allocation ratios must sum to 1.")
        confidence = self._calculate_decision_confidence(recommendation, intervention)
        cost, service, mitigation = self._calculate_expected_outcomes(
            action, suppliers, allocations, market_conditions
        )
        decision = SourcingDecision(
            decision_id="decision_{}".format(len(self.partnership_history)),
            component_name=component_name,
            final_action=action,
            selected_suppliers=suppliers,
            allocation_ratios=allocations,
            ai_recommendation=recommendation,
            human_intervention=intervention,
            decision_confidence=confidence,
            expected_cost=cost,
            expected_service_level=service,
            risk_mitigation=mitigation,
            made_at=datetime.now(),
        )
        self.partnership_history.append(decision)
        return decision

    def _assess_intervention_need(
        self, recommendation: AIRecommendation
    ) -> Tuple[bool, Optional[HumanExpert]]:
        """Apply confidence, risk, and trust rules to decide whether an expert reviews."""
        if recommendation.confidence_score < 0.6 or recommendation.risk_assessment > 0.7:
            return True, self._select_best_expert(recommendation)
        for expert in self.experts.values():
            if expert.trust_in_ai < expert.intervention_threshold and expert.collaboration_score > 0.6:
                return True, expert
        return False, None

    def _select_best_expert(self, recommendation: AIRecommendation) -> HumanExpert:
        """Prefer a strategic expert for highly critical components; otherwise use collaboration score."""
        component = next(
            (item for item in self.components if item["name"] == recommendation.component_name), None
        )
        if component is not None and component["criticality_score"] > 0.8:
            strategic = [item for item in self.experts.values()
                         if item.expertise_domain == "strategic"]
            if strategic:
                return max(strategic, key=lambda item: item.collaboration_score)
        return max(self.experts.values(), key=lambda item: item.collaboration_score)

    def _get_human_intervention(
        self, recommendation: AIRecommendation, expert: HumanExpert
    ) -> HumanIntervention:
        """Create a simulated accept, modify, override, or escalate record."""
        kind = self._simulate_intervention_type(recommendation, expert)
        modified_action, modified_supplier = None, None
        if kind in ("modify", "override"):
            modified_action, modified_supplier = self._simulate_modification(recommendation, expert)
        reasoning = self._generate_intervention_reasoning(
            recommendation, expert, modified_action
        ) if modified_action else (
            "AI recommendation accepted after expert review."
            if kind == "accept" else "Further review is required because of the assessed risk or impact."
        )
        confidence = recommendation.confidence_score if kind == "accept" else (
            expert.confidence_level if kind in ("modify", "override") else 0.5
        )
        return HumanIntervention(
            intervention_id="intervention_{}".format(len(self.partnership_history)),
            expert_id=expert.expert_id,
            recommendation_id=recommendation.recommendation_id,
            intervention_type=kind,
            reasoning=reasoning,
            modified_action=modified_action,
            modified_supplier=modified_supplier,
            confidence_in_decision=confidence,
            timestamp=datetime.now(),
        )

    def _simulate_intervention_type(self, recommendation: AIRecommendation, expert: HumanExpert) -> str:
        """Simulate expert response using trust, AI confidence, style, and expertise match."""
        accept_probability = expert.trust_in_ai * 0.7
        if recommendation.confidence_score > 0.8:
            accept_probability += 0.2
        elif recommendation.confidence_score < 0.6:
            accept_probability -= 0.3
        if expert.decision_style == "conservative" and recommendation.risk_assessment > 0.6:
            accept_probability -= 0.3
        elif expert.decision_style == "aggressive":
            accept_probability += 0.2
        component = next(
            (item for item in self.components if item["name"] == recommendation.component_name), None
        )
        if (component is not None and component["criticality_score"] > 0.8
                and expert.expertise_domain == "strategic"):
            accept_probability += 0.1
        accept_probability = float(np.clip(accept_probability, 0.0, 1.0))
        draw = np.random.random()
        if draw < accept_probability:
            return "accept"
        if draw < min(1.0, accept_probability + 0.3):
            return "modify"
        if draw < min(1.0, accept_probability + 0.5):
            return "override"
        return "escalate"

    def _simulate_modification(
        self, recommendation: AIRecommendation, expert: HumanExpert
    ) -> Tuple[str, str]:
        """Apply the expert-style rule while retaining a supplier from the recommendation."""
        if expert.decision_style == "conservative":
            component = next(
                item for item in self.components if item["name"] == recommendation.component_name
            )
            if recommendation.action_type != "dual_sourcing" and component["criticality_score"] > 0.7:
                return "dual_sourcing", recommendation.primary_supplier
            return recommendation.action_type, recommendation.primary_supplier
        if expert.decision_style == "aggressive" and recommendation.action_type == "dual_sourcing":
            return "single_sourcing", recommendation.primary_supplier
        return recommendation.action_type, recommendation.primary_supplier

    def _generate_intervention_reasoning(
        self, recommendation: AIRecommendation, expert: HumanExpert, modified_action: str
    ) -> str:
        """State the simulated rationale for the expert's action."""
        if expert.decision_style == "conservative" and modified_action == "dual_sourcing":
            return "Risk mitigation is prioritized for a critical component."
        if expert.decision_style == "aggressive" and modified_action == "single_sourcing":
            return "The expert prioritizes cost efficiency while retaining the recommended supplier."
        return "The expert balances cost efficiency and risk management using the available recommendation."

    def _update_trust(
        self, expert: HumanExpert, recommendation: AIRecommendation, intervention: HumanIntervention
    ) -> None:
        """Update bounded trust and collaboration indicators after the simulated review."""
        if intervention.intervention_type == "accept":
            expert.trust_in_ai = min(0.99, expert.trust_in_ai + 0.05 * recommendation.confidence_score)
            expert.collaboration_score = min(0.99, expert.collaboration_score + 0.03)
        elif intervention.intervention_type in ("modify", "override"):
            expert.trust_in_ai = max(0.3, expert.trust_in_ai - 0.02)
            expert.collaboration_score = min(0.99, expert.collaboration_score + 0.05)
        else:
            expert.trust_in_ai = max(0.2, expert.trust_in_ai - 0.05)
            expert.collaboration_score = max(0.3, expert.collaboration_score - 0.02)
        self.trust_metrics[expert.expert_id].append({
            "timestamp": datetime.now(),
            "trust_in_ai": expert.trust_in_ai,
            "collaboration_score": expert.collaboration_score,
            "intervention_type": intervention.intervention_type,
        })

    def _calculate_decision_confidence(
        self, recommendation: AIRecommendation, intervention: Optional[HumanIntervention]
    ) -> float:
        """Combine AI confidence with the simulated expert response."""
        if intervention is None:
            return recommendation.confidence_score
        if intervention.intervention_type == "accept":
            return min(0.99, recommendation.confidence_score * 1.1)
        if intervention.intervention_type in ("modify", "override"):
            return (recommendation.confidence_score + intervention.confidence_in_decision) / 2.0
        return max(0.3, recommendation.confidence_score * 0.7)

    def _calculate_expected_outcomes(
        self, action: str, suppliers: List[str], allocations: List[float], market_conditions: Dict
    ) -> Tuple[float, float, float]:
        """Calculate unit cost, service estimate, and risk-mitigation score for the final allocation."""
        supplier_data = {item["name"]: item for item in self.suppliers}
        selected = [supplier_data[name] for name in suppliers]
        average_cost = sum(item["unit_cost"] * share for item, share in zip(selected, allocations))
        if action == "dual_sourcing" and len(selected) == 2:
            reliability = sum(
                item["reliability_score"] * share for item, share in zip(selected, allocations)
            )
            quality = max(item["quality_score"] for item in selected)
            service = min(0.98, reliability * quality * 1.1)
            mitigation = 0.85
        else:
            service = selected[0]["reliability_score"] * selected[0]["quality_score"]
            mitigation = 0.6
        return average_cost, service, mitigation

    def record_decision_outcome(self, decision: SourcingDecision, actual_outcome: Dict) -> None:
        """Record a simulated result and pass the observed measures to the learning rule."""
        actual_cost = actual_outcome.get("actual_cost", decision.expected_cost)
        actual_service = actual_outcome.get("actual_service", decision.expected_service_level)
        performance = actual_outcome.get("performance_score", 0.8)
        cost_variance = (actual_cost - decision.expected_cost) / max(abs(decision.expected_cost), 1e-12)
        service_variance = actual_service - decision.expected_service_level
        record = {
            "decision_id": decision.decision_id,
            "timestamp": datetime.now(),
            "actual_cost": actual_cost,
            "actual_service": actual_service,
            "actual_risk_mitigation": actual_outcome.get(
                "actual_risk_mitigation", decision.risk_mitigation
            ),
            "performance_score": performance,
            "cost_variance": cost_variance,
            "service_variance": service_variance,
        }
        self.decision_outcomes.append(record)
        baseline_cost = np.mean([item["unit_cost"] for item in self.suppliers])
        observed_cost_impact = (actual_cost - baseline_cost) / baseline_cost
        self.ai_engine.learn_from_outcome(decision.ai_recommendation, {
            "cost_impact": observed_cost_impact,
            "service_level": actual_service,
            "performance_score": performance,
            "actual_cost": actual_cost,
            "actual_confidence": decision.decision_confidence,
        })
        if decision.human_intervention is not None:
            self._update_expert_learning(decision.human_intervention, record)
        self.learning_loop.record_learning_event("decision_outcome", {
            "performance_score": performance,
            "decision_id": decision.decision_id,
        })
        self._calculate_partnership_roi(record)

    def _update_expert_learning(self, intervention: HumanIntervention, outcome: Dict) -> None:
        """Adjust expert confidence and intervention threshold from the recorded outcome."""
        expert = self.experts[intervention.expert_id]
        score = outcome["performance_score"]
        if score > 0.8:
            expert.confidence_level = min(0.99, expert.confidence_level + 0.02)
            if intervention.intervention_type == "accept":
                expert.trust_in_ai = min(0.99, expert.trust_in_ai + 0.03)
        elif score < 0.6:
            expert.confidence_level = max(0.4, expert.confidence_level - 0.03)
            if intervention.intervention_type in ("modify", "override"):
                expert.trust_in_ai = max(0.3, expert.trust_in_ai - 0.02)
        if intervention.intervention_type == "accept" and score < 0.6:
            expert.intervention_threshold = max(0.5, expert.intervention_threshold - 0.05)

    def _calculate_partnership_roi(self, outcome: Dict) -> None:
        """Compute the example's illustrative ROI index from performance, cost, and service."""
        performance_bonus = (outcome["performance_score"] - 0.7) * 100
        cost_savings = max(0.0, -outcome["cost_variance"]) * 50
        service_improvement = max(0.0, outcome["service_variance"]) * 30
        self.partnership_roi.append({
            "timestamp": datetime.now(),
            "roi_value": performance_bonus + cost_savings + service_improvement,
            "performance_score": outcome["performance_score"],
            "cost_variance": outcome["cost_variance"],
            "service_variance": outcome["service_variance"],
        })

    def get_partnership_metrics(self) -> PartnershipMetrics:
        """Aggregate current trust, collaboration, outcomes, interventions, learning, and ROI."""
        average_trust = float(np.mean([item.trust_in_ai for item in self.experts.values()]))
        average_collaboration = float(np.mean([
            item.collaboration_score for item in self.experts.values()
        ]))
        recent = self.decision_outcomes[-5:]
        quality = float(np.mean([item["performance_score"] for item in recent])) if recent else 0.7
        total = len(self.partnership_history)
        intervention_count = sum(
            decision.human_intervention is not None for decision in self.partnership_history
        )
        intervention_frequency = intervention_count / max(1, total)
        successful = sum(
            outcome["performance_score"] > 0.7
            for decision in self.partnership_history if decision.human_intervention is not None
            for outcome in self.decision_outcomes if outcome["decision_id"] == decision.decision_id
        )
        intervention_success = successful / max(1, intervention_count)
        accuracy = self.ai_engine.performance_metrics["confidence_accuracy"]
        if len(accuracy) > 10:
            recent_accuracy = np.mean(accuracy[-5:])
            early_accuracy = np.mean(accuracy[:5])
            learning_progress = (recent_accuracy - early_accuracy) / max(0.1, early_accuracy)
        else:
            learning_progress = 0.5
        if len(self.partnership_roi) > 5:
            roi_score = float(np.clip(np.mean([
                item["roi_value"] for item in self.partnership_roi[-5:]
            ]) / 100.0, 0.0, 1.0))
        else:
            roi_score = 0.5
        return PartnershipMetrics(
            timestamp=datetime.now(), trust_level=average_trust,
            collaboration_effectiveness=average_collaboration,
            decision_quality_score=quality, intervention_frequency=intervention_frequency,
            intervention_success_rate=intervention_success,
            learning_progress=learning_progress, roi_score=roi_score,
        )

    def generate_partnership_report(self) -> str:
        """Create a readable summary of this simulated partnership run."""
        metrics = self.get_partnership_metrics()
        report = [
            "# Human-AI Symbiotic Partnership Report",
            "Generated: {}".format(metrics.timestamp.strftime("%Y-%m-%d %H:%M:%S")),
            "",
            "## Partnership Overview",
            "- Total decisions: {}".format(len(self.partnership_history)),
            "- Human experts: {}".format(len(self.experts)),
            "",
            "## Key Metrics",
            "- Trust level: {:.1%}".format(metrics.trust_level),
            "- Collaboration effectiveness: {:.1%}".format(metrics.collaboration_effectiveness),
            "- Decision quality score: {:.1%}".format(metrics.decision_quality_score),
            "- Intervention frequency: {:.1%}".format(metrics.intervention_frequency),
            "- Intervention success rate: {:.1%}".format(metrics.intervention_success_rate),
            "- Learning progress indicator: {:.1%}".format(metrics.learning_progress),
            "- Illustrative ROI score: {:.1%}".format(metrics.roi_score),
            "",
            "## Expert Performance",
        ]
        for expert in self.experts.values():
            report.append(
                "- {} ({}): trust {:.1%}, collaboration {:.1%}, confidence {:.1%}, style {}".format(
                    expert.name, expert.expertise_domain, expert.trust_in_ai,
                    expert.collaboration_score, expert.confidence_level, expert.decision_style,
                )
            )
        if self.decision_outcomes:
            recent = self.decision_outcomes[-5:]
            report.extend([
                "",
                "## Recent Outcome Averages",
                "- Performance score: {:.1%}".format(np.mean([
                    item["performance_score"] for item in recent
                ])),
                "- Cost variance: {:+.1%}".format(np.mean([
                    item["cost_variance"] for item in recent
                ])),
                "- Service variance: {:+.1%}".format(np.mean([
                    item["service_variance"] for item in recent
                ])),
            ])
        report.extend([
            "",
            "## Interpretation",
            "These indicators summarize simulated records and illustrative scoring rules. They are not validated estimates of business ROI or causal effects.",
        ])
        return "\n".join(report)


class LearningLoop:
    """Store outcome events and summarize observed performance trends."""

    def __init__(self):
        self.learning_history = []
        self.performance_trends = defaultdict(list)

    def record_learning_event(self, event_type: str, data: Dict) -> None:
        """Save an event and its performance score when one is supplied."""
        self.learning_history.append({
            "timestamp": datetime.now(), "event_type": event_type, "data": data
        })
        if "performance_score" in data:
            self.performance_trends[event_type].append(data["performance_score"])

    def analyze_learning_progress(self) -> Dict:
        """Compare the first and most recent five observations for sufficiently long series."""
        result = {
            "total_events": len(self.learning_history),
            "event_types": sorted(set(item["event_type"] for item in self.learning_history)),
            "performance_trends": {},
        }
        for event_type, scores in self.performance_trends.items():
            if len(scores) > 5:
                recent = float(np.mean(scores[-5:]))
                early = float(np.mean(scores[:5]))
                result["performance_trends"][event_type] = {
                    "improvement": (recent - early) / max(0.1, early),
                    "recent_performance": recent,
                    "early_performance": early,
                }
        return result

In [ ]:
def create_partnership_example():
    """Return the MediTech component, supplier, and expert example data."""
    components_data = [
        {
            "name": "Component_1_Critical", "criticality_score": 0.9,
            "current_inventory": 230.0, "safety_stock": 46.0,
            "demand_forecast": 11.5, "current_supplier": "Supplier_A",
        },
        {
            "name": "Component_2_Standard", "criticality_score": 0.6,
            "current_inventory": 405.0, "safety_stock": 81.0,
            "demand_forecast": 20.25, "current_supplier": "Supplier_B",
        },
        {
            "name": "Component_3_Medium", "criticality_score": 0.75,
            "current_inventory": 160.0, "safety_stock": 32.0,
            "demand_forecast": 6.15, "current_supplier": "Supplier_C",
        },
    ]
    suppliers_data = [
        {"name": "Supplier_A", "unit_cost": 15.0, "reliability_score": 0.85,
         "quality_score": 0.87, "capacity_utilization": 0.75},
        {"name": "Supplier_B", "unit_cost": 18.0, "reliability_score": 0.92,
         "quality_score": 0.94, "capacity_utilization": 0.65},
        {"name": "Supplier_C", "unit_cost": 16.0, "reliability_score": 0.88,
         "quality_score": 0.90, "capacity_utilization": 0.80},
        {"name": "Supplier_D", "unit_cost": 14.0, "reliability_score": 0.82,
         "quality_score": 0.85, "capacity_utilization": 0.70},
    ]
    experts_data = [
        {"id": "expert_001", "name": "Dr. Sarah Chen", "domain": "strategic",
         "experience": 15, "confidence": 0.85, "style": "balanced",
         "initial_trust": 0.6, "intervention_threshold": 0.7},
        {"id": "expert_002", "name": "Michael Rodriguez", "domain": "operational",
         "experience": 12, "confidence": 0.78, "style": "conservative",
         "initial_trust": 0.5, "intervention_threshold": 0.6},
        {"id": "expert_003", "name": "Lisa Thompson", "domain": "financial",
         "experience": 10, "confidence": 0.82, "style": "aggressive",
         "initial_trust": 0.7, "intervention_threshold": 0.8},
    ]
    return components_data, suppliers_data, experts_data


components_data, suppliers_data, experts_data = create_partnership_example()
partnership = HumanAIPartnership(components_data, suppliers_data, experts_data)

print("Human-AI Partnership initialized with:")
print("  {} components".format(len(components_data)))
print("  {} suppliers".format(len(suppliers_data)))
print("  {} human experts".format(len(experts_data)))
print("\nExperts:")
for expert in partnership.experts.values():
    print("  - {} ({}, {} style)".format(
        expert.name, expert.expertise_domain, expert.decision_style
    ))

In [ ]:
weekly_decisions = []
weekly_metrics = []

print("=== 12-WEEK PARTNERSHIP SIMULATION ===")
for week in range(1, 13):
    print("\n--- Week {} ---".format(week))
    market_conditions = {
        "market_index": np.random.normal(0.0, 0.3),
        "disruption_risk": max(0.05, np.random.normal(0.15, 0.1)),
    }
    week_decisions = []
    for component in components_data:
        decision = partnership.make_collaborative_decision(component["name"], market_conditions)
        week_decisions.append(decision)
        actual_outcome = {
            "actual_cost": decision.expected_cost * np.random.normal(1.0, 0.1),
            "actual_service": min(
                0.98, decision.expected_service_level * np.random.normal(1.0, 0.05)
            ),
            "actual_risk_mitigation": decision.risk_mitigation * np.random.normal(1.0, 0.1),
            "performance_score": np.random.beta(2, 1) * 0.3 + 0.7,
        }
        partnership.record_decision_outcome(decision, actual_outcome)
        intervention_type = (
            decision.human_intervention.intervention_type
            if decision.human_intervention else "None"
        )
        expert_name = (
            partnership.experts[decision.human_intervention.expert_id].name
            if decision.human_intervention else "AI only"
        )
        supplier_summary = ", ".join(
            "{} ({:.0%})".format(name, share)
            for name, share in zip(decision.selected_suppliers, decision.allocation_ratios)
        )
        print("  {}: {} — {}".format(component["name"], decision.final_action, supplier_summary))
        print("    Confidence: {:.2f}; intervention: {} by {}".format(
            decision.decision_confidence, intervention_type, expert_name
        ))
        print("    Expected unit cost: ${:.2f}; service estimate: {:.1%}".format(
            decision.expected_cost, decision.expected_service_level
        ))
    weekly_decisions.append(week_decisions)
    metrics = partnership.get_partnership_metrics()
    weekly_metrics.append(metrics)
    print("  Weekly-to-date trust: {:.1%}; collaboration: {:.1%}; quality: {:.1%}".format(
        metrics.trust_level, metrics.collaboration_effectiveness,
        metrics.decision_quality_score
    ))
    print("  Intervention frequency: {:.1%}; illustrative ROI score: {:.1%}".format(
        metrics.intervention_frequency, metrics.roi_score
    ))

print("\n=== SIMULATION COMPLETED ===")
final_report = partnership.generate_partnership_report()
print(final_report)

assert len(weekly_metrics) == 12
assert len(partnership.partnership_history) == 12 * len(components_data)
for decision in partnership.partnership_history:
    assert np.isclose(sum(decision.allocation_ratios), 1.0)
    assert len(decision.selected_suppliers) == len(decision.allocation_ratios)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle("Human-AI Partnership: Simulated Metrics", fontsize=16, fontweight="bold")
weeks = list(range(1, len(weekly_metrics) + 1))
trust_levels = [item.trust_level for item in weekly_metrics]
collaboration_scores = [item.collaboration_effectiveness for item in weekly_metrics]
decision_qualities = [item.decision_quality_score for item in weekly_metrics]
intervention_rates = [item.intervention_frequency for item in weekly_metrics]
roi_scores = [item.roi_score for item in weekly_metrics]
learning_progress = [item.learning_progress for item in weekly_metrics]

plots = [
    (axes[0, 0], trust_levels, "Trust in AI over time", "Mean trust score", 0.7, "Reference level"),
    (axes[0, 1], collaboration_scores, "Collaboration effectiveness", "Mean collaboration score", 0.8, "Reference level"),
    (axes[0, 2], decision_qualities, "Decision quality over time", "Mean performance score", 0.85, "Reference level"),
    (axes[1, 0], intervention_rates, "Human intervention frequency", "Share of decisions reviewed", 0.3, "Reference level"),
    (axes[1, 1], roi_scores, "Illustrative ROI score", "Normalized score", 0.7, "Reference level"),
]
for axis, values, title, ylabel, reference, reference_label in plots:
    axis.plot(weeks, values, marker="o", linewidth=2, label="Observed simulation metric")
    axis.axhline(reference, color="red", linestyle="--", alpha=0.6, label=reference_label)
    axis.set_xlabel("Week")
    axis.set_ylabel(ylabel)
    axis.set_title(title)
    axis.set_ylim(0, 1)
    axis.grid(True, alpha=0.3)
    axis.legend()

ax = axes[1, 2]
expert_names = [expert.name for expert in partnership.experts.values()]
final_trust = [expert.trust_in_ai for expert in partnership.experts.values()]
final_collaboration = [expert.collaboration_score for expert in partnership.experts.values()]
final_confidence = [expert.confidence_level for expert in partnership.experts.values()]
positions = np.arange(len(expert_names))
width = 0.25
ax.bar(positions - width, final_trust, width, label="Trust in AI")
ax.bar(positions, final_collaboration, width, label="Collaboration")
ax.bar(positions + width, final_confidence, width, label="Confidence")
ax.set_xlabel("Expert")
ax.set_ylabel("Score")
ax.set_title("Final expert indicators")
ax.set_xticks(positions)
ax.set_xticklabels([name.split()[0] for name in expert_names], rotation=35, ha="right")
ax.set_ylim(0, 1)
ax.grid(True, axis="y", alpha=0.3)
ax.legend()
for axis in axes.flat:
    axis.set_axisbelow(True)
plt.tight_layout()
plt.show()

print("Learning-progress indicator is plotted as a simulation metric; its values are relative changes when enough outcomes are available.")
print("Observed weekly values:", len(weeks), "weeks")

In [ ]:
def analyze_intervention_patterns():
    """Summarize intervention types, their observed outcomes, and expert-level records."""
    print("=== INTERVENTION PATTERN ANALYSIS ===")
    intervention_counts = defaultdict(int)
    intervention_scores = defaultdict(list)
    outcomes_by_id = {item["decision_id"]: item for item in partnership.decision_outcomes}
    for decision in partnership.partnership_history:
        intervention = decision.human_intervention
        if intervention is not None:
            intervention_counts[intervention.intervention_type] += 1
            outcome = outcomes_by_id.get(decision.decision_id)
            if outcome is not None:
                intervention_scores[intervention.intervention_type].append(
                    outcome["performance_score"]
                )
    total = sum(intervention_counts.values())
    print("\nIntervention frequency:")
    if total == 0:
        print("  No expert interventions were recorded in this run.")
    for kind, count in intervention_counts.items():
        print("  {}: {} ({:.1f}% of interventions)".format(
            kind.title(), count, 100.0 * count / total
        ))
    print("\nObserved performance by intervention type:")
    for kind, scores in intervention_scores.items():
        if scores:
            print("  {}: mean {:.3f}, standard deviation {:.3f}, n={}".format(
                kind.title(), np.mean(scores), np.std(scores), len(scores)
            ))
    print("\nExpert-specific records:")
    for expert in partnership.experts.values():
        decisions = [
            item for item in partnership.partnership_history
            if item.human_intervention is not None
            and item.human_intervention.expert_id == expert.expert_id
        ]
        scores = [outcomes_by_id[item.decision_id]["performance_score"]
                  for item in decisions if item.decision_id in outcomes_by_id]
        if decisions:
            score_summary = "{:.3f}".format(np.mean(scores)) if scores else "not available"
            print("  {}: {} reviews; mean recorded performance {}".format(
                expert.name, len(decisions), score_summary
            ))


def analyze_learning_progress():
    """Report recorded AI and expert learning indicators without asserting causal improvement."""
    print("\n=== LEARNING PROGRESS ANALYSIS ===")
    accuracy = partnership.ai_engine.performance_metrics["confidence_accuracy"]
    if len(accuracy) > 10:
        early = float(np.mean(accuracy[:5]))
        recent = float(np.mean(accuracy[-5:]))
        improvement = (recent - early) / max(0.1, early)
        print("\nAI confidence-alignment indicator:")
        print("  Early mean: {:.3f}".format(early))
        print("  Recent mean: {:.3f}".format(recent))
        print("  Relative change: {:+.1%}".format(improvement))
        print("\nCurrent decision weights:")
        print("  Cost: {:.3f}".format(partnership.ai_engine.cost_weight))
        print("  Service: {:.3f}".format(partnership.ai_engine.service_weight))
        print("  Risk: {:.3f}".format(partnership.ai_engine.risk_weight))
    else:
        print("AI confidence-alignment comparison requires more than 10 recorded outcomes.")
    initial_experts = {item["id"]: item for item in experts_data}
    print("\nHuman expert indicators:")
    for expert in partnership.experts.values():
        initial = initial_experts[expert.expert_id]
        print("  {}:".format(expert.name))
        print("    Trust: {:.3f} to {:.3f}".format(
            initial.get("initial_trust", 0.5), expert.trust_in_ai
        ))
        print("    Collaboration: 0.500 to {:.3f}".format(expert.collaboration_score))
        print("    Confidence: {:.3f} to {:.3f}".format(
            initial["confidence"], expert.confidence_level
        ))


def calculate_partnership_roi():
    """Break down the example's illustrative ROI index and its early/recent trend."""
    print("\n=== PARTNERSHIP ROI ANALYSIS ===")
    records = partnership.partnership_roi
    if not records:
        print("No outcome records are available.")
        return
    total_roi = sum(item["roi_value"] for item in records)
    print("Illustrative total ROI index: {:.2f}".format(total_roi))
    print("Mean index per decision: {:.2f}".format(total_roi / len(records)))
    performance_value = sum(max(0.0, (item["performance_score"] - 0.7) * 100)
                            for item in records)
    cost_value = sum(max(0.0, -item["cost_variance"] * 50) for item in records)
    service_value = sum(max(0.0, item["service_variance"] * 30) for item in records)
    print("\nIllustrative index components:")
    print("  Performance component: {:.2f}".format(performance_value))
    print("  Cost component: {:.2f}".format(cost_value))
    print("  Service component: {:.2f}".format(service_value))
    if len(records) > 6:
        early = np.mean([item["roi_value"] for item in records[:6]])
        recent = np.mean([item["roi_value"] for item in records[-6:]])
        change = (recent - early) / max(abs(early), 1e-12)
        print("\nRecent versus early mean index change: {:+.1%}".format(change))
    print("This index is defined by illustrative scaling rules and is not monetary ROI.")


analyze_intervention_patterns()
analyze_learning_progress()
calculate_partnership_roi()

print("\n=== FINAL PARTNERSHIP ASSESSMENT ===")
final_metrics = partnership.get_partnership_metrics()
status = (
    "MEETS SIMULATED THRESHOLDS"
    if final_metrics.trust_level > 0.7 and final_metrics.roi_score > 0.6
    else "BELOW ONE OR MORE SIMULATED THRESHOLDS"
)
overall_score = (
    final_metrics.trust_level + final_metrics.collaboration_effectiveness
    + final_metrics.decision_quality_score + final_metrics.roi_score
) / 4.0
print("Partnership status: {}".format(status))
print("Mean of four reported indicators: {:.1%}".format(overall_score))
print("Interpret these labels as summaries of this simulated run, not validated business conclusions.")

## Strengths, limitations, and appropriate use

### Strengths of the human–AI approach

- **Explainable recommendations:** Each selected action includes a score-related rationale and reported cost, service, and risk indicators.
- **Human oversight:** The intervention protocol creates explicit opportunities for expert review.
- **Adaptive records:** Outcomes inform performance tracking, and the specified learning rule can adjust cost and service weights.
- **Context-sensitive decisions:** Component criticality, market conditions, and expert attributes influence the simulated process.

### Limitations

- The supplier scores, confidence formula, intervention probabilities, outcome distributions, and ROI scaling are illustrative assumptions rather than empirically validated estimates.
- The process does not solve a constrained sourcing optimization problem. Its recommendation is a ranking among the candidate options generated by the stated rules.
- A simulated expert response is not a substitute for measured human judgment. Likewise, a simulated outcome does not establish that an intervention caused a better or worse result.
- Supplier dependence, correlated disruptions, capacity limits, demand uncertainty, lead times, and inventory replenishment are not represented in sufficient detail for operational deployment.
- Because the simulation uses random draws without a fixed seed, repeated runs can produce different metrics and recommendations.

### Appropriate use

This example is useful for studying how an explainable sourcing heuristic, an intervention protocol, trust indicators, and outcome tracking can be connected in one workflow. It is most appropriate as a conceptual teaching demonstration. Applying the framework to a real sourcing decision would require validated data, explicit business constraints, calibrated scoring rules, governance, and evaluation against observed outcomes.